# 01 — Data, EDA, FFM estimation and evolution of $\hat r$

Thesis chapters 3–4 and Section 5.1. This notebook

1. loads the (trading day × 5-min) log-return matrices built by `scripts/build_dataset.py`,
2. builds the volatility proxy $\log(r^2 + \varepsilon_i)$ and smooths each daily curve with penalised B-splines,
3. runs the exploratory analysis (descriptive statistics, correlation structure, Gram spectrum, Frobenius heatmap),
4. estimates the Functional Factor Model on 2014–2025 and selects the number of factors (IC1a, ER/GR, ABC),
5. tracks $\hat r$ and the leading Gram eigenvalue over rolling windows,
6. **saves the shared objects** (`data/shared/*.pkl`) used by notebooks `02_forecasting`, `03_stability` and `04_raw_ffm`.

## Contents

1. [Imports](#Imports)
2. [Data](#Data)
3. [EDA](#EDA)
4. [FFM 2014–2025](#FFM-2014-2025) — number of factors, goodness of fit, ABC
5. [Rolling window](#Rolling-window) — evolution of $\hat r$, leading eigenvalue
6. [Save shared objects](#Save-shared-objects)
7. [Missing values — all candidate pairs](#Missing-values-—-all-candidate-pairs) (pair selection)

# Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

In [ ]:
# ── Standard library ─────────────────────────────────────────────────────────
import os
from functools import partial

In [ ]:
# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.stattools import acf

In [ ]:
# ── Data processing & factor model ───────────────────────────────────────────
from data_processing import panel_to_dict
from ffm_transform import log_transform
from ffm_smoothing import smoothing_for_ffm, extract_smoothing_params
from ffm_fx_real import run_ffm
from factor_model_fcts import build_panel_from_matrices, compute_daily_avg_vol, ffm_per_period
from functions import plot_heatmap_grid

In [ ]:
# ── EDA ───────────────────────────────────────────────────────────────────────
from ffm_eda import (
    missing_by_year_pair,
    get_stable_pairs,
    plot_raw_vs_smooth,
    plot_abs_returns_over_time,
)
from ffm_eda_focused import run_focused_eda

In [ ]:
# ── Forecasting ───────────────────────────────────────────────────────────────

In [ ]:
# ── Stability ─────────────────────────────────────────────────────────────────

In [ ]:
%load_ext autoreload
%autoreload 2

# Data

Raw HistData / Dukascopy files are downloaded and converted into (trading day × 5-min) log-return
matrices by the scripts in `scripts/` (see [`data/README.md`](../data/README.md)).
This notebook starts from those matrices: `data/<pair>_matrix_lg.parquet` (in-sample) and
`data/2026/<pair>_2026_matrix_lg.parquet` (out-of-sample, used for forecasting).

## In-sample panel (2014–2025)

In [ ]:
FX_all = ['audcad', 'audchf', 'audjpy', 'audnzd', 'cadchf', 'cadjpy', 'chfjpy', 'euraud', 'eurcad', 'eurchf', 
      'eurgbp', 'eurjpy', 'eurnok', 'eurnzd', 'eursek', 'eurusd', 'gbpaud', 'gbpcad', 'gbpchf', 'gbpjpy', 'gbpnzd', 
      'gbpusd', 'nzdcad', 'nzdchf', 'nzdjpy', 'nzdusd', 'sgdjpy', 'usdcad', 'usdchf',  'usdjpy',
      'usdnok', 'usdsek', 'usdsgd', 'audusd', 'eurczk',
       # 'eurhuf',
          'eurpln', 
      #'eurtry', 
      'usdczk', 'usdhuf', 
     'usdpln', 'usdtry', 'usdmxn', 'usdzar', 'zarjpy', 'usdcnh']

In [ ]:
path = "../data/"

vol_matrices_all = {}

for i,fx in enumerate(FX_all):

    df = pd.read_parquet(f"{path}{fx}_matrix_lg.parquet")

    vol_matrices_all[fx] = df
    print(f"{fx} loaded — start: {df.index[0]} | end: {df.index[-1]}")

In [ ]:
first_pair = list(vol_matrices_all.values())[0]
print(f"Date range : {first_pair.index[0]} → {first_pair.index[-1]}")
print(f"Raw number of days : {len(first_pair)}")
print(f"Number of pairs    : {len(vol_matrices_all)}")


In [ ]:
vol_matrices_2014 = {k: v.loc['2014-01-01':] for k, v in vol_matrices_all.items()}

vol_matrices_2014_week  = {k: v[pd.to_datetime(v.index).dayofweek < 5] for k, v in vol_matrices_2014.items()}
vol_ffm_all = smoothing_for_ffm(vol_matrices_2014_week,smooth=True,n_basis=20, drop_sparse=True, max_nan_pct=0.70)

In [ ]:
plot_abs_returns_over_time(
    vol_matrices_2014,
    pairs = ['eurusd', 'audjpy', 'eurchf', 'usdtry'],
)

In [ ]:
vol_sq_all  = {
    k: v[pd.to_datetime(v.index).dayofweek < 5] ** 2
    for k, v in vol_matrices_2014.items()
}
vol_log_all = log_transform(vol_sq_all)

## Out-of-sample panel (2026)

Smoothing parameters ($\varepsilon_i$, $\lambda_i$) are estimated on the training sample and re-used on the 2026 data.

In [ ]:
path = "../data/2026/"

vol_matrices_2026 = {}

for i,fx in enumerate(FX_all):

    df = pd.read_parquet(f"{path}{fx}_2026_matrix_lg.parquet")

    vol_matrices_2026[fx] = df
    print(f"{fx} loaded — start: {df.index[0]} | end: {df.index[-1]}")

In [ ]:
params = extract_smoothing_params(vol_matrices_2014, n_basis=20)
print(params)

In [ ]:

vol_ffm_2026 = smoothing_for_ffm(
    vol_matrices_2026,
    smooth=True,
    n_basis=20,
    eps_dict=params['eps'],
    lam_dict=params['lam'],
)

# EDA

## Stats 

In [ ]:
stats = pd.DataFrame({
    pair: pd.Series(df.values.flatten()).dropna().describe()
    for pair, df in vol_ffm_all.items()
}).T

# Round and rename the columns for LaTeX export
stats_latex = stats[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']].copy()
stats_latex.columns = ['N', 'Mean', 'Std', 'Min', 'Q1', 'Median', 'Q3', 'Max']

print(stats_latex.to_latex(
    float_format="{:.4f}".format,
    caption="Descriptive statistics of intraday volatility by FX pair",
    label="tab:vol_stats",
    column_format="l" + "r" * 8
))


In [ ]:
all_vals = np.concatenate([df.values.flatten() for df in vol_ffm_all.values()])
all_vals = all_vals[~np.isnan(all_vals)]
print(pd.Series(all_vals).describe())

global_stats = pd.Series(all_vals).describe().rename("All pairs")
print(global_stats.to_frame().T.to_latex(
    float_format="{:.4f}".format,
    caption="Global descriptive statistics of intraday volatility",
    label="tab:vol_stats_global"
))


## Covariance/correlation heatmap

In [ ]:
from functions import plot_mxm_heatmaps, plot_pxp_heatmap

In [ ]:
pairs = ['eurusd', 'audjpy', 'eurchf', 'usdtry']

In [ ]:
plot_mxm_heatmaps(
    vol_ffm_all,
    pairs,
    correlation=True,
    lag1=False,
    start_date='2014-01-01',
    cmap="RdBu_r",
    figsize=(20, 16),
    save_dir='../figures/EDA',
)

In [ ]:
plot_pxp_heatmap(
    vol_ffm_all,
    correlation=True,
    start_date='2014-01-01',
    end_date=None,
    cmap="RdBu_r",
    figsize=(10, 8),
    save_dir='../figures/EDA',
)

## Spectral analysis

In [ ]:
from functions import print_gram_eigenvalues

print_gram_eigenvalues(vol_ffm_all, n_components=10, start_date='2014-01-01')

## Frobenius Heatmap

In [ ]:
from functions import plot_semester_covariance_heatmaps

# with vol_clean (after filter_session)
#plot_semester_covariance_heatmaps(vol_clean)

# or with vol_ready (after smoothing + log-transform)
plot_semester_covariance_heatmaps(vol_ffm_all)

In [ ]:
vol_matrices_2014_clean, valid_days, tau, pairs = build_panel_from_matrices(vol_matrices_2014, min_coverage=0.7)

col_names = list(next(iter(vol_matrices_2014.values())).columns)

result_dict = panel_to_dict(vol_matrices_2014_clean, valid_days, pairs, col_names=col_names)



In [ ]:
vol_squared = {k: v**2 for k, v in result_dict.items()}

run_focused_eda(
    vol_ffm_all,
    selected=['eurusd', 'audjpy', 'eurchf', 'usdtry'],
    data_label='smoothed_log',
    output_dir='../figures/EDA'
)

In [ ]:
plot_heatmap_grid(
    pairs=[vol_ffm_all['eurusd'], vol_ffm_all['audjpy'],
           vol_ffm_all['eurchf'], vol_ffm_all['usdtry']],
    titles=['EUR/USD', 'AUD/JPY', 'EUR/CHF', 'USD/TRY'],
    start_date='2014-01-01',
    ytick_step=90,
)

In [ ]:
plot_raw_vs_smooth(
    vol_log_all,
    vol_ffm_all,
    day   = '2025-12-18',
    pairs = ['eurusd', 'audjpy', 'eurchf', 'usdtry'],)

In [ ]:
stable_pairs = get_stable_pairs(vol_matrices_all, '2014-01-01', '2025-12-31')

print("Stable pairs:", stable_pairs)
print('Instable pairs:', [p for p in vol_matrices_all.keys() if p not in stable_pairs])

A pair is considered *stable* if at least 80% of its trading days have at least 70% of their intraday intervals filled, over the whole 2014–2025 period.

# FFM 2014-2025

## Smoothing

### Full sample 2014–2025

In [ ]:
ffm = run_ffm(vol_ffm_all, k_max=5, min_coverage=0.7
              ,date_start = '2014-01-01', 
              #date_end = '2025-12-31'
              silent = False
              )

In [ ]:
Y_ffm = ffm['X'] - ffm['mu'][:, np.newaxis, :]   # (N, T, M)
Chi   = ffm['Chi_hat']                              # (N, T, M)

# Frobenius R² (natural goodness-of-fit of the factor model)
r2_frob = (np.linalg.norm(Chi)**2) / (np.linalg.norm(Y_ffm)**2)
print(f"R² Frobenius : {r2_frob:.1%}")

# Mean cross-sectional R² (explained cross-pair variance)
r2_cs = np.mean(
    np.var(Chi, axis=0) / (np.var(Y_ffm, axis=0) + 1e-14)
)
print(f"R² cross-sectionnel moyen : {r2_cs:.1%}")


In [ ]:
pairs = ffm['pairs']
Chi   = ffm['Chi_hat']          # (N, T, M)
Y_ffm = ffm['X'] - ffm['mu'][:, np.newaxis, :]

r2_per_pair = pd.Series(
    {pairs[i]: np.linalg.norm(Chi[i])**2 / np.linalg.norm(Y_ffm[i])**2
     for i in range(len(pairs))}
).sort_values(ascending=False)

print(r2_per_pair.to_string(float_format='{:.1%}'.format))
print(f"\nMoyenne : {r2_per_pair.mean():.1%}")


### Sub-period 2017–2025

Estimated on 2017–2025 only, the eigenvalue ratio selects $\hat r = 2$ — a first sign of instability (thesis, Section 5.1).

In [ ]:
ffm_2017 = run_ffm(vol_ffm_all, k_max=5, min_coverage=0.7, date_start='2017-01-01', silent=True)
print('r_hat (ER), 2017-2025 =', ffm_2017['r_hat'])

In [ ]:
avg_vol = compute_daily_avg_vol(
    ffm['X'],
    ffm['days'],
    log_space=False   # returns exp(x/2) ≈ mean |return|
)

# Sanity check
print(avg_vol.head())
print(f"Shape : {avg_vol.shape}")   # (T,)

# Plot rapide
avg_vol.plot(figsize=(14, 4), title="Daily avg vol — all FX pairs")
plt.ylabel("|r| mean × 10⁴")
plt.tight_layout()
print(avg_vol.nlargest(20))

## ABC method

In [ ]:
import importlib, sys
import abc_method
importlib.reload(abc_method)
from abc_method import abc_select_r, plot_stability_curves, abc_per_period
# Y = centred panel (N, T, M) — output of run_ffm
Y = ffm['Y']   # shape (N, T, M)
print(f"Panel shape : {Y.shape}  (N pairs × T days × M intervals)")

In [ ]:
#c_grid = np.logspace(-2, 0.801, 300)   # 0.001 a 2, log-espace
c_grid =np.linspace(0,5, 300)

r_hat, r_hat_per_perm, c_star_list, c_grid, var_curves, r_curves = abc_select_r(
    Y,
    k_max       = 10,
    c_grid      = c_grid,
    P           = 5,
    n_subpanels = 10, #J 
    frac_min    = 0.8,
    seed        = 42,
    verbose     = True,
)

print(f"r̂ (median over permutations) = {r_hat}")
print(f"r̂ per permutation             : {r_hat_per_perm}")
print(f"optimal ĉ per permutation      : {c_star_list.round(3)}")

In [ ]:
abc_results = {
    'r_hat'         : r_hat,
    'r_hat_per_perm': r_hat_per_perm,
    'c_star_list'   : c_star_list,
    'c_grid'        : c_grid,
    'var_curves'    : var_curves,
    'r_curves'      : r_curves,
}

In [ ]:
plot_stability_curves(
    var_curves,
    c_grid,
    c_star_list,
    r_hat_per_perm,
    r_hat,
    r_curves  = r_curves,
    save_path = "../figures/STABILITY/abc_stability_curves.png",
)

# Rolling window

## Evolution of $\hat r$

### IC1a, ER, GR and ABC on rolling windows

In [ ]:
# Rolling estimate of r over 6-month windows (main text) and 3 / 12-month windows (appendix):
# IC1a, ER, GR from the rolling FFM, ABC from abc_per_period
c_grid = np.linspace(0, 5, 300)
ends   = {3: '2025-10-01', 6: '2025-07-01', 12: '2025-01-01'}

for W in [3, 6, 12]:
    print(f"Running FFM for windows of {W} months...")
    df_periods = ffm_per_period(
        data        = vol_ffm_all,
        start_date  = '2014-01-01',
        end_date    = ends[W],
        n_months    = W,
        func        = lambda x: run_ffm(x, k_max=5, min_coverage=0.7, silent=True),
        step_months = 1,
    )
    df_abc = abc_per_period(
        data        = vol_ffm_all,
        start_date  = '2014-01-01',
        end_date    = ends[W],
        n_months    = W,
        step_months = 1,
        k_max       = 10,
        P           = 3,
        c_grid      = c_grid,
    )
    abc_tmp = df_abc[['start', 'r_hat_abc']].copy()
    abc_tmp['start'] = pd.to_datetime(abc_tmp['start'])
    df_periods['start'] = pd.to_datetime(df_periods['start'])
    df_periods = df_periods.drop(columns='r_hat_abc', errors='ignore').merge(abc_tmp, on='start', how='left')
    df_periods.to_csv(f"../data/results/ffm_results_by_period_{W}M.csv", index=False)

### Plot

In [ ]:
W = 6   # main text; use 3 or 12 for the appendix figures
df_periods = pd.read_csv(f"../data/results/ffm_results_by_period_{W}M.csv", parse_dates=['start'])

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

dates = pd.to_datetime(df_periods['start'])

# ── Colours and styles ────────────────────────────────────────────────
STYLES = {
    'IC1a': dict(color='#D7191C', lw=2.5,  ls='-',  marker='o', ms=5,  zorder=5, label='IC1a'),
    'ER'  : dict(color='#2C7BB6', lw=2.0,  ls='--', marker='s', ms=4,  zorder=4, label='Spectral ratio (ER)'),
    'GR'  : dict(color='#19CA2B', lw=1.5,  ls=':',  marker='^', ms=3,  zorder=3, label='GR', alpha=0.85),
    'ABC' : dict(color='#FF7F00', lw=2.5,  ls='-',  marker='D', ms=5,  zorder=6, label='ABC'),
}

SERIES = {
    'IC1a': df_periods['r_hat_IC1a'],
    'ER'  : df_periods['r_hat_ratio'],
    'GR'  : df_periods['r_hat_GR'],
    'ABC' : df_periods['r_hat_abc'],
}

# ── Background and grid ───────────────────────────────────────────────
ax.set_facecolor('#FAFAFA')
ax.yaxis.grid(True, color='#E0E0E0', linewidth=1.2, zorder=0)
ax.set_axisbelow(True)

# ── Crisis windows ────────────────────────────────────────────────────
crises = [
    ('2015-01-14', 'CHF floor\nremoval'),
    ('2018-08-10', 'TRY crisis'),
    ('2016-06-24', 'Brexit vote'),
    ('2020-03-08', 'COVID-19'),
    ('2022-02-24', 'Ukraine war'),
]
for d, label in crises:
    ax.axvspan(pd.to_datetime(d) - pd.Timedelta(days=30),
               pd.to_datetime(d) + pd.Timedelta(days=30),
               alpha=0.08, color='#888', zorder=1)

# ── Curves ────────────────────────────────────────────────────────────
# Small vertical offset to avoid overlapping lines when r̂=1
offsets = {'IC1a': 0.0, 'ER': 0.1, 'GR': 0.2, 'ABC': 0.3}

for key, s in SERIES.items():
    style  = STYLES[key]
    y      = s + offsets[key]
    mask   = y.notna()  # ignore NaN (periods without ABC estimate)
    ax.step(dates[mask], y[mask], where='post',
            color=style['color'], linewidth=style['lw'],
            linestyle=style['ls'], zorder=style['zorder'],
            alpha=style.get('alpha', 1.0), label=style['label'])
    ax.scatter(dates[mask], y[mask],
               color=style['color'], s=style['ms']**2,
               zorder=style['zorder'] + 1,
               alpha=style.get('alpha', 1.0))

# ── Reference line r=1 ────────────────────────────────────────────────
ax.axhline(1, color='#AAAAAA', linewidth=1.0, linestyle='-', zorder=0)
ax.text(dates.iloc[-1] + pd.Timedelta(days=10), 1.0,
        'r=1', fontsize=9, color='#AAAAAA', va='center')

# ── Axes and labels ───────────────────────────────────────────────────
r_max = max(s.dropna().max() for s in SERIES.values())
ax.set_ylim(0.6, r_max + 0.8)
ax.set_yticks(range(1, int(r_max) + 2))
ax.set_yticklabels([f'r={i}' for i in range(1, int(r_max) + 2)], fontsize=10)
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Estimated number of factors r̂', fontsize=11)
ax.set_title(
    'Factor count r̂ over time — ER vs IC1a vs GR vs ABC\n'
    '12-month rolling windows  |  N=44 FX pairs  |  2014–2025',
    fontsize=13, fontweight='bold', pad=12
)

ax.legend(loc='upper left', fontsize=10, framealpha=0.9,
          edgecolor='#DDD', fancybox=False)

for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
ax.spines['left'].set_color('#DDD')
ax.spines['bottom'].set_color('#DDD')
ax.tick_params(axis='x', rotation=30)

# Annotate the crisis windows once the final y-limits are known
for d, label in crises:
    ax.text(pd.to_datetime(d), r_max + 0.6,
            label, fontsize=8, ha='center', va='top',
            color='#666', style='italic')

plt.tight_layout()
plt.savefig(f'../figures/STABILITY/r_hat_comparison_{W}M.png', dpi=180, bbox_inches='tight',
            facecolor='white')
plt.show()


## Leading eigenvalue (rolling)

In [ ]:
from ffm_stability import plot_eigenvalue_rolling

period = [3,6,12]
 
for p in period: 
    if p == 3: end = '2025-10-01'
    if p == 6: end = '2025-07-01'
    if p == 12: end ='2025-01-01'
    df = plot_eigenvalue_rolling(
        vol_ffm_all,
        window_size=p,
        step_months=1,
        start_date='2014-01-01',
        end_date=end,
    )

In [ ]:
print(df.tail())

# Save shared objects

Save the objects needed by notebooks `02_forecasting`, `03_stability` and `04_raw_ffm`.
Files are written to `data/shared/` (git-ignored).

In [ ]:
abc_results = {
    'r_hat'         : r_hat,
    'r_hat_per_perm': r_hat_per_perm,
    'c_star_list'   : c_star_list,
    'c_grid'        : c_grid,
    'var_curves'    : var_curves,
}

In [ ]:
# ── Save shared objects for the forecasting & stability notebooks ─────────────
import joblib, os

shared_dir = "../data/shared/"
os.makedirs(shared_dir, exist_ok=True)

joblib.dump(ffm,                f"{shared_dir}ffm.pkl")
joblib.dump(vol_ffm_all,        f"{shared_dir}vol_ffm_all.pkl")
joblib.dump(vol_matrices_all,   f"{shared_dir}vol_matrices_all.pkl")
joblib.dump(vol_matrices_2026,  f"{shared_dir}vol_matrices_2026.pkl")
joblib.dump(vol_ffm_2026,       f"{shared_dir}vol_ffm_2026.pkl")
joblib.dump(avg_vol,            f"{shared_dir}avg_vol.pkl")
joblib.dump(FX_all,             f"{shared_dir}FX_all.pkl")
joblib.dump(abc_results,        f"{shared_dir}abc_results.pkl")

print("Shared objects saved to", shared_dir)
print(f"ffm — T={ffm['U_hat'].shape[1]} days, N={len(ffm['pairs'])} pairs")

# Missing values — all candidate pairs

Missing-value rates per year for all 46 candidate pairs (before the pair selection).

In [ ]:
FX_all_all = ['audcad', 'audchf', 'audjpy', 'audnzd', 'cadchf', 'cadjpy', 'chfjpy', 'euraud', 'eurcad', 'eurchf', 
      'eurgbp', 'eurjpy', 'eurnok', 'eurnzd', 'eursek', 'eurusd', 'gbpaud', 'gbpcad', 'gbpchf', 'gbpjpy', 'gbpnzd', 
      'gbpusd', 'nzdcad', 'nzdchf', 'nzdjpy', 'nzdusd', 'sgdjpy', 'usdcad', 'usdchf',  'usdjpy',
      'usdnok', 'usdsek', 'usdsgd', 'audusd', 'eurczk','eurhuf',
    'eurpln', 'eurtry', 
      'usdczk', 'usdhuf', 
     'usdpln', 'usdtry', 'usdmxn', 'usdzar', 'zarjpy', 'usdcnh']

In [ ]:
path = "../data/"

temp = {}

for i,fx in enumerate(FX_all_all):

    df = pd.read_parquet(f"{path}{fx}_matrix_lg.parquet")

    temp[fx] = df
    print(f"{fx} loaded — start: {df.index[0]} | end: {df.index[-1]}")

In [ ]:
# ── Missing values per year and pair ──────────────────────────────────────────
df_missing = missing_by_year_pair(temp)

# Console display (1 decimal)
with pd.option_context('display.max_columns', None,
                       'display.width',       200,
                       'display.float_format', '{:.1f}%'.format):
    print(df_missing)

# Export CSV
#df_missing.to_csv("../figures/EDA/missing_pct_by_year_pair.csv")

# ── Heatmap ───────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(max(12, len(df_missing.columns) * 0.45),
                                max(6,  len(df_missing) * 0.4)))

im = ax.imshow(df_missing.values, aspect='auto', cmap='Reds',
               vmin=0, vmax=df_missing.values[np.isfinite(df_missing.values)].max())

ax.set_xticks(range(len(df_missing.columns)))
ax.set_xticklabels(df_missing.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(df_missing.index)))
ax.set_yticklabels(df_missing.index.astype(int), fontsize=8)
ax.set_xlabel('Pair')
ax.set_ylabel('Year')
ax.set_title('Missing values (%) — raw data, before imputation')

# Value in each cell
for i in range(len(df_missing.index)):
    for j in range(len(df_missing.columns)):
        val = df_missing.iloc[i, j]
        if np.isfinite(val):
            ax.text(j, i, f'{val:.0f}', ha='center', va='center',
                    fontsize=6, color='black' if val < 30 else 'white')
        else:
            ax.text(j, i, '—', ha='center', va='center', fontsize=7, color='grey')

plt.colorbar(im, ax=ax, label='% NaN')
plt.tight_layout()
plt.savefig('../figures/EDA/missing_heatmap_all.png', dpi=150)
plt.show()